# 🏋️ Day 3 실습 — LCEL · TypedDict · Pydantic · Structured Output

📖 강의 연계: Day 3 강의교안 전체 범위 (모듈 3-1 ~ 3-4)

✅ **완료 기준**
- [ ] LCEL 파이프를 직접 조립해 `invoke`·`batch` 실행
- [ ] TypedDict 선언 후 런타임 동작과 에디터 경고 차이 확인
- [ ] LLM 응답을 Pydantic 인스턴스로 받아 `.field` 접근
- [ ] 마이 서비스 조각 Pydantic 스키마 + LangSmith 링크 제출

⚠️ 우측 상단 커널이 **`.venv`** 인지 확인하세요 (Colab 아님)

In [15]:
# 환경 점검 — 이 셀이 "환경 준비 완료"를 출력해야 다음 셀로 진행합니다
from dotenv import load_dotenv
import os

load_dotenv()

assert os.getenv("OPENAI_API_KEY"), (
    "❌ OPENAI_API_KEY 없음 — .env 파일 확인 (📖 강의 교안 모듈 1-4 참조)"
)
if not os.getenv("LANGCHAIN_API_KEY"):
    print("⚠️  LANGCHAIN_API_KEY 없음 — LangSmith 트레이스가 기록되지 않습니다")

print("✅ 환경 준비 완료")
print(f"   API 키 앞 7자: {os.getenv('OPENAI_API_KEY')[:7]}...")

✅ 환경 준비 완료
   API 키 앞 7자: sk-proj...


In [16]:
# LLM 공통 초기화 — 이후 모든 Step에서 재사용합니다
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
print("✅ LLM 초기화 완료:", llm.model_name)

✅ LLM 초기화 완료: gpt-4o-mini


---
## Step 1. LCEL 파이프라인 — `prompt | llm | parser`

📖 강의 연계: day3 강의교안 **모듈 3-1** "Runnable & LCEL 파이프라인"

LCEL `|` 파이프는 앞 단계 출력을 다음 단계 입력으로 자동 연결합니다.
세 단계의 데이터 타입 변환을 직접 확인합니다:

```
입력 dict → ChatPromptTemplate → list[Message] → ChatOpenAI → AIMessage → StrOutputParser → str
```

In [17]:
# Step 1-① 그대로 실행 — 강의에서 본 체인을 실행해보세요
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

parser = StrOutputParser()   # AIMessage → str 변환기

prompt = ChatPromptTemplate.from_messages([
    ("system", "당신은 요약 전문가입니다."),
    ("human", "다음 텍스트를 {length}줄로 요약해:\n\n{text}")
])

chain = prompt | llm | parser   # LCEL 파이프 조립

sample_text = """
2024년 AI 캠퍼스 프로젝트 킥오프 회의가 오후 2시에 진행되었습니다.
주요 안건은 팀 구성, 프로젝트 범위, 일정 계획이었습니다.
팀은 총 6개 팀으로 구성되며 각 팀당 4명으로 편성됩니다.
최종 발표는 8월 14일 오후 4시에 진행될 예정입니다.
모든 팀원은 8월 7일까지 서비스 설계서를 제출해야 합니다.
"""

result = chain.invoke({"length": "3", "text": sample_text})

print("✅ type(result):", type(result))   # <class 'str'>
print()
print(result)

✅ type(result): <class 'langchain_core.messages.base.TextAccessor'>

2024년 AI 캠퍼스 프로젝트 킥오프 회의가 오후 2시에 열렸으며, 팀 구성, 프로젝트 범위, 일정 계획이 주요 안건이었습니다. 총 6개 팀이 각 4명으로 편성되며, 최종 발표는 8월 14일 오후 4시에 예정되어 있습니다. 모든 팀원은 8월 7일까지 서비스 설계서를 제출해야 합니다.


In [18]:
# Step 1-② 한 곳만 바꾸기 — 요약 줄 수만 바꿔서 차이를 관찰하세요
# TODO(🔰): 아래 "___" 를 숫자로 채우세요 (예: "1", "5", "7")
# 힌트: 📖 강의 교안 모듈 3-1 "v1 — 가장 단순한 형태" 예시 참고

result_v2 = chain.invoke({"length": "___", "text": sample_text})  # ← ___ 채우기
print(result_v2)

2024년 AI 캠퍼스 프로젝트 킥오프 회의가 오후 2시에 열렸으며, 팀은 6개로 구성되고 각 팀은 4명으로 편성됩니다. 최종 발표는 8월 14일 오후 4시에 있으며, 서비스 설계서는 8월 7일까지 제출해야 합니다.


In [19]:
# Step 1-③ batch — 여러 텍스트를 동시에 처리합니다
texts = [
    "오늘 오전 팀 회의에서 8월 14일 발표 일정을 확정했습니다.",
    "새 프로젝트 예산이 승인되어 팀 구성과 장비 구매를 시작합니다.",
    "고객사로부터 요구사항 변경 요청이 들어왔습니다. 다음 주 중 검토 예정입니다.",
]

results = chain.batch([
    {"length": "2", "text": texts[0]},
    {"length": "1", "text": texts[1]},
    {"length": "3", "text": texts[2]},
])

for i, r in enumerate(results):
    print(f"\n=== 텍스트 {i+1} 요약 ({[2,1,3][i]}줄) ===")
    print(r)

print("\n✅ type(results):", type(results), "/ 길이:", len(results))


=== 텍스트 1 요약 (2줄) ===
오늘 오전 팀 회의에서 8월 14일 발표 일정이 확정되었습니다. 관련 사항을 준비해 주시기 바랍니다.

=== 텍스트 2 요약 (1줄) ===
새 프로젝트의 예산이 승인되어 팀 구성과 장비 구매를 시작합니다.

=== 텍스트 3 요약 (3줄) ===
고객사에서 요구사항 변경 요청이 있었습니다. 이 요청은 다음 주 중에 검토될 예정입니다. 필요한 조치를 취할 계획입니다.

✅ type(results): <class 'list'> / 길이: 3


---
## Step 1-④ LCEL 실행 방식 완전 정리 — stream / batch / ainvoke

📖 강의 연계: day3 강의교안 **모듈 3-1** "LCEL 실행 방식 한눈에 보기"

| 메서드 | 동작 | 주요 용도 |
|--------|------|----------|
| `invoke` | 완성될 때까지 기다린 후 결과 반환 | 단건 처리, 후속 처리가 필요한 경우 |
| `stream` | 토큰이 생성되는 즉시 출력 | 챗봇 UI 타이핑 효과 (파이프라인 모듈에서 본격 활용) |
| `batch` | 여러 입력을 한 번에 병렬 처리 | 대량 데이터 전처리·일괄 분류 |
| `ainvoke` / `astream` / `abatch` | 위 3종의 비동기 버전 | FastAPI·웹 서버 (파이프라인 모듈) |

> ⚠️ **노트북에서 비동기**: Jupyter는 이미 이벤트 루프가 실행 중이므로  
> `asyncio.run()`은 사용 불가 — `await` 를 셀에 직접 씁니다.  
> `.py` 파일로 이식할 때는 `asyncio.run()`으로 바꿉니다.

In [20]:
# Step 1-④ stream + max_concurrency batch
# stream: 토큰이 생성될 때마다 즉시 출력합니다 — 타이핑 효과

print("=== stream: 토큰 단위 실시간 출력 ===")
for chunk in chain.stream({"length": "3", "text": sample_text}):
    print(chunk, end='', flush=True)   # end='' 로 줄바꿈 없이 이어붙임
print("\n")

# batch max_concurrency — Rate Limit 초과 방어
results_limited = chain.batch(
    [{"length": "2", "text": t} for t in texts],
    config={"max_concurrency": 3},   # 동시 요청 최대 3개 (기본값 5)
)
print(f"✅ batch(max_concurrency=3): {len(results_limited)}건 완료")
print("   파이프라인 모듈의 asyncio.Semaphore 와 같은 역할을 batch 수준에서 수행합니다")

=== stream: 토큰 단위 실시간 출력 ===
2024년 AI 캠퍼스 프로젝트 킥오프 회의가 오후 2시에 열렸으며, 팀 구성, 프로젝트 범위, 일정 계획이 주요 안건이었습니다. 총 6개 팀이 각 4명으로 편성되며, 최종 발표는 8월 14일 오후 4시에 예정되어 있습니다. 모든 팀원은 8월 7일까지 서비스 설계서를 제출해야 합니다.

✅ batch(max_concurrency=3): 3건 완료
   파이프라인 모듈의 asyncio.Semaphore 와 같은 역할을 batch 수준에서 수행합니다


---
## ⭐ Step 1-⭐ Runnable 4종 완전 정복

📖 강의 연계: day3 강의교안 **모듈 3-1** "⭐ 심화: Runnable 4종 완전 정복"

| 종류 | 역할 | 주요 사용 시점 |
|------|------|--------------|
| `RunnablePassthrough` | 입력을 변환 없이 그대로 통과 | `chain.invoke("문자열")` 직접 입력 가능하게 할 때 |
| `RunnableLambda` | 파이썬 함수를 Runnable로 감싸기 | 체인 중간 커스텀 전처리·후처리 |
| `RunnableParallel` | 동일 입력을 여러 체인에 동시 실행 | context + question 동시 준비 (RAG 패턴) |
| `RunnableSequence` | 순차 파이프라인 명시적 정의 | `|` 연산자와 동일한 결과 |

> 💡 **10월 LangGraph 복선**: `itemgetter`는 LangGraph State 딕셔너리에서  
> 특정 키만 꺼내 프롬프트에 전달할 때 동일하게 사용합니다.

In [21]:
# ⭐ Runnable 4종 전체 예시 — 직접 실행해서 출력을 확인하세요
from langchain_core.runnables import (
    RunnablePassthrough, RunnableLambda,
    RunnableParallel, RunnableSequence,
)
from operator import itemgetter

# ─── ① RunnablePassthrough — 문자열 직접 입력 패턴 ──────────────────────
# {"question": RunnablePassthrough()}: invoke("질문") 시 {"question": "질문"} 자동 변환
prompt_q  = ChatPromptTemplate.from_template("이 질문에 간단히 답해줘: {question}")
chain_rpt = {"question": RunnablePassthrough()} | prompt_q | llm | StrOutputParser()

print("① RunnablePassthrough — 문자열 직접 입력:")
print(chain_rpt.invoke("파이썬 가상환경이 뭐야?"))   # 문자열 직접 입력!
# chain_rpt.invoke({"question": "파이썬 가상환경이 뭐야?"}) 와 동일

# ─── ② RunnableLambda — 파이썬 함수를 체인 중간에 삽입 ─────────────────
preprocess = RunnableLambda(lambda x: x.strip().upper())   # 공백 제거 + 대문자
greeting   = RunnableLambda(lambda name: f"안녕하세요, {name}님!")

chain_rl = preprocess | greeting
print("\n② RunnableLambda — 전처리 체인:")
print(chain_rl.invoke("  alice  "))   # "안녕하세요, ALICE님!"

# ─── ③ itemgetter — 딕셔너리에서 특정 키만 추출 ────────────────────────
# 10월 LangGraph State 딕셔너리 패턴의 복선
prompt_ig = ChatPromptTemplate.from_template(
    "{고객번호} 고객님, {창구번호}번 창구로 오십시오."
)
chain_ig = (
    {
        "고객번호": itemgetter("customer_number"),   # dict["customer_number"] 추출
        "창구번호": itemgetter("counter_number"),
    }
    | prompt_ig | llm | StrOutputParser()
)
print("\n③ itemgetter — 딕셔너리 키 추출:")
print(chain_ig.invoke({"customer_number": "132", "counter_number": "4"}))

# ─── ④ RunnableSequence — | 연산자와 완전히 동일 ───────────────────────
double    = RunnableLambda(lambda x: x + x)
say_hello = RunnableLambda(lambda name: f"Hello, {name}!")
seq       = RunnableSequence(first=double, last=say_hello)

print("\n④ RunnableSequence vs | 연산자 (결과 동일):")
print(seq.invoke("Minjae"))                      # "Hello, MinjaeMinjae!"
print((double | say_hello).invoke("Minjae"))     # 동일 출력

① RunnablePassthrough — 문자열 직접 입력:
파이썬 가상환경은 프로젝트마다 독립적인 파이썬 실행 환경을 만들어주는 도구입니다. 이를 통해 각 프로젝트에 필요한 패키지와 라이브러리를 별도로 관리할 수 있어, 서로 다른 프로젝트 간의 의존성 충돌을 방지할 수 있습니다. 주로 `venv`나 `virtualenv` 같은 도구를 사용하여 생성합니다.

② RunnableLambda — 전처리 체인:
안녕하세요, ALICE님!

③ itemgetter — 딕셔너리 키 추출:
안녕하세요! 132 고객님, 4번 창구로 가시면 됩니다. 도움이 필요하시면 언제든지 말씀해 주세요!

④ RunnableSequence vs | 연산자 (결과 동일):
Hello, MinjaeMinjae!
Hello, MinjaeMinjae!


---
## Step 2. TypedDict — 딕셔너리에 이름표만 붙이기

📖 강의 연계: day3 강의교안 **모듈 3-2** "핵심 개념 ①: TypedDict"

TypedDict는 딕셔너리에 타입 라벨만 붙입니다. 런타임에는 **그냥 dict** 입니다.
검증은 하지 않지만, VS Code 에디터가 키 오타를 잡아줍니다.

> 💡 **안내판(TypedDict)** vs **검문소(Pydantic)** — 오늘 두 차이를 직접 확인합니다.

In [22]:
# Step 2-① 그대로 실행 — TypedDict 선언 후 접근
from typing import TypedDict

class ChatState(TypedDict):
    question:    str
    answer:      str
    tokens_used: int

# 생성·사용은 그냥 딕셔너리와 동일
state: ChatState = {"question": "LCEL이란?", "answer": "", "tokens_used": 0}
state["answer"] = "LangChain Expression Language입니다."
state["tokens_used"] = 42

print("✅ TypedDict는 런타임엔 그냥 dict:")
print(f"   type(state): {type(state)}")     # <class 'dict'>
print(f"   question:    {state['question']}")
print(f"   tokens_used: {state['tokens_used']}")
print()
print("💡 VS Code에서 state[\'  '] 를 입력할 때 키가 자동 완성되나요?")

✅ TypedDict는 런타임엔 그냥 dict:
   type(state): <class 'dict'>
   question:    LCEL이란?
   tokens_used: 42

💡 VS Code에서 state['  '] 를 입력할 때 키가 자동 완성되나요?


In [23]:
# Step 2-② 한 곳만 바꾸기 — 의도적으로 잘못된 키를 넣어보세요
# TODO(🔰): 아래 "___" 에 ChatState에 없는 키를 입력해보세요
# 예시: "tokns"(오타), "Token"(대소문자), "score"(없는 키) 등
# 힌트: 런타임 에러는 없습니다 — 하지만 VS Code에서 노란 밑줄 경고가 보이면 성공!

state["tokn"] = 99   # TODO: 잘못된 키를 넣어보세요

print("⚠️  런타임 에러 없음 — TypedDict는 실행 시 검증하지 않습니다")
print(f"   state 내용: {state}")

⚠️  런타임 에러 없음 — TypedDict는 실행 시 검증하지 않습니다
   state 내용: {'question': 'LCEL이란?', 'answer': 'LangChain Expression Language입니다.', 'tokens_used': 42, 'tokn': 99}


In [24]:
# Step 2-③ TypedDict vs Pydantic 런타임 차이 비교
from pydantic import BaseModel

class ChatStatePydantic(BaseModel):
    question:    str
    answer:      str
    tokens_used: int

print("=== TypedDict: 런타임 검증 없음 ===")
bad_state: ChatState = {"question": 123, "answer": "", "tokens_used": "열 개"}  # 오류 없음!
print(f"  잘못된 타입도 그냥 저장됨: {bad_state}")

print()
print("=== Pydantic: 런타임 ValidationError ===")
try:
    bad_pydantic = ChatStatePydantic(
        question="안녕?", answer="", tokens_used="열 개"  # int여야 하는데 str
    )
except Exception as e:
    error_line = str(e).split("\n")[1] if "\n" in str(e) else str(e)
    print(f"  ❌ ValidationError! {error_line.strip()}")

good = ChatStatePydantic(question="안녕?", answer="안녕하세요!", tokens_used=42)
print(f"\n✅ 올바른 Pydantic 접근: good.tokens_used = {good.tokens_used}")
print(f"   (딕셔너리 아님 → 점 표기법  good.tokens_used, type: {type(good).__name__})")

=== TypedDict: 런타임 검증 없음 ===
  잘못된 타입도 그냥 저장됨: {'question': 123, 'answer': '', 'tokens_used': '열 개'}

=== Pydantic: 런타임 ValidationError ===
  ❌ ValidationError! tokens_used

✅ 올바른 Pydantic 접근: good.tokens_used = 42
   (딕셔너리 아님 → 점 표기법  good.tokens_used, type: ChatStatePydantic)


---
## Step 3. Pydantic BaseModel — 검사까지 하는 이름표

📖 강의 연계: day3 강의교안 **모듈 3-2** "핵심 개념 ②: Pydantic BaseModel"

`Field(description=...)`이 AI에게 주는 지시입니다.  
모호하면 엉뚱한 값, 명확하면 정확한 값이 들어옵니다.

In [25]:
# Step 3-① 그대로 실행 — TaskClassification 완성 예시
from pydantic import BaseModel, Field
from typing import Literal, Optional

class TaskClassification(BaseModel):
    """업무 분류 모델 — AI가 이 필드들을 채웁니다"""
    category: Literal["기술지원", "구매요청", "일정조율", "기타"] = Field(
        description="업무 유형 4가지 중 하나. 반드시 이 4가지 중에서 선택."
    )
    priority: int = Field(
        description="우선순위 1(낮음)~5(높음)",
        ge=1, le=5
    )
    summary: str = Field(
        description="핵심 요약 20자 이내",
        max_length=20
    )
    urgent: bool = Field(description="24시간 이내 처리가 필요하면 True")
    assignee: Optional[str] = Field(
        default=None, description="담당 부서명. 불명확하면 None"
    )

# 올바른 데이터 생성
task = TaskClassification(
    category="기술지원", priority=4, summary="VPN 연결 불가", urgent=True
)
print(f"✅ category:  {task.category}")
print(f"   priority: {task.priority}")
print(f"   urgent:   {task.urgent}")
print(f"   assignee: {task.assignee}")
print(f"\n📦 model_dump(): {task.model_dump()}")

✅ category:  기술지원
   priority: 4
   urgent:   True
   assignee: None

📦 model_dump(): {'category': '기술지원', 'priority': 4, 'summary': 'VPN 연결 불가', 'urgent': True, 'assignee': None}


In [26]:
# Step 3-② 한 곳만 바꾸기 — deadline 선택 필드를 추가하세요
# TODO(🔰): 아래 MyTaskClassification에 'deadline' 필드를 추가하세요
#  - 타입:     Optional[str]
#  - 기본값:   None
#  - description: "처리 기한 (YYYY-MM-DD 형식). 없으면 None"
# 힌트: 📖 강의 교안 모듈 3-2 "BaseModel 기본 문법" → assignee 필드 참고

class MyTaskClassification(BaseModel):
    category: Literal["기술지원", "구매요청", "일정조율", "기타"] = Field(
        description="업무 유형 4가지 중 하나"
    )
    priority: int  = Field(description="우선순위 1~5", ge=1, le=5)
    summary:  str  = Field(description="20자 이내 요약", max_length=20)
    urgent:   bool = Field(description="24시간 이내 처리 여부")
    # TODO(🔰): deadline 필드를 여기에 추가하세요 ↓
    deadline: Optional[str] = Field(default="2026-09-09", description="처리 기한 (YYYY-MM-DD 형식)")

t = MyTaskClassification(category="구매요청", priority=3,
                         summary="노트북 구매 요청", urgent=False, deadline="2026-09-123") #chatstatepydantic이 아니라서 오류가 있어도 그냥 출력됨
print(t.model_dump())

{'category': '구매요청', 'priority': 3, 'summary': '노트북 구매 요청', 'urgent': False, 'deadline': '2026-09-123'}


In [27]:
# Step 3-③ ValidationError 의도적 발생 — 에러 메시지를 직접 읽어보세요
print("=== ValidationError 실험 ===\n")

test_cases = [
    {"이름": "category 오류 (Literal 위반)",
     "data": {"category": "없는분류", "priority": 3, "summary": "테스트", "urgent": False}},
    {"이름": "priority 오류 (ge/le 위반)",
     "data": {"category": "기술지원", "priority": 10, "summary": "테스트", "urgent": False}},
    {"이름": "summary 오류 (max_length 초과)",
     "data": {"category": "기술지원", "priority": 3, "summary": "아" * 25, "urgent": False}},
]

for case in test_cases:
    try:
        TaskClassification(**case["data"])
        print(f"✅ {case['이름']}: 통과")
    except Exception as e:
        first_error = str(e).split("\n")
        short = next((l.strip() for l in first_error if "Input should" in l or
                      "less than" in l or "String should" in l), str(e)[:80])
        print(f"❌ {case['이름']}")
        print(f"   오류 내용: {short}")
        print()

=== ValidationError 실험 ===

❌ category 오류 (Literal 위반)
   오류 내용: Input should be '기술지원', '구매요청', '일정조율' or '기타' [type=literal_error, input_value='없는분류', input_type=str]

❌ priority 오류 (ge/le 위반)
   오류 내용: Input should be less than or equal to 5 [type=less_than_equal, input_value=10, input_type=int]

❌ summary 오류 (max_length 초과)
   오류 내용: String should have at most 20 characters [type=string_too_long, input_value='아아아아아아아아...아아아아아아아', input_type=str]



---
## Step 4. with_structured_output — LLM 출력을 Pydantic 인스턴스로

📖 강의 연계: day3 강의교안 **모듈 3-3** "with_structured_output"

`llm.with_structured_output(스키마)` — LLM에게 "반드시 이 형식으로만 답해라"는 계약을 맺습니다.  
반환 타입이 `str`에서 **Pydantic 인스턴스**로 바뀌는 것을 직접 확인합니다.

In [28]:
# Step 4-① 그대로 실행 — EmailSummary 완성 체인
from langchain_core.prompts import ChatPromptTemplate
from typing import Optional

class EmailSummary(BaseModel):
    sender:       str           = Field(description="발신자 이름")
    purpose:      str           = Field(description="이메일 목적 한 문장")
    action_items: list[str]     = Field(description="처리 필요 항목 목록")
    deadline:     Optional[str] = Field(default=None, description="기한. 없으면 None")
    priority:     int           = Field(description="중요도 1~5", ge=1, le=5)

structured_llm   = llm.with_structured_output(EmailSummary)
prompt_template  = ChatPromptTemplate.from_messages([
    ("system", "이메일 분석 전문가입니다. 요청된 필드를 정확하게 추출하세요."),
    ("human", "다음 이메일을 분석해주세요:\n\n{email}"),
])
chain_struct = prompt_template | structured_llm

test_email = """
안녕하세요, 이팀장님.

9/12(수) 오전 10시 MCP 프로젝트 킥오프 미팅 참석 부탁드립니다.
준비 자료: 팀 소개 슬라이드 3~5장
확인 후 9/10(토)까지 회신 부탁드립니다.

홍길동 드림
"""

result = chain_struct.invoke({"email": test_email})

print(f"✅ type(result): {type(result).__name__}")   # EmailSummary (str 아님!)
print(f"\n   sender:       {result.sender}")
print(f"   purpose:      {result.purpose}")
print(f"   action_items: {result.action_items}")
print(f"   deadline:     {result.deadline}")
print(f"   priority:     {result.priority}")
print(f"\n📦 model_dump(): {result.model_dump()}")

✅ type(result): EmailSummary

   sender:       홍길동
   purpose:      MCP 프로젝트 킥오프 미팅 참석 요청
   action_items: ['MCP 프로젝트 킥오프 미팅 참석', '팀 소개 슬라이드 3~5장 준비', '9/10(토)까지 회신']
   deadline:     2023-09-10
   priority:     3

📦 model_dump(): {'sender': '홍길동', 'purpose': 'MCP 프로젝트 킥오프 미팅 참석 요청', 'action_items': ['MCP 프로젝트 킥오프 미팅 참석', '팀 소개 슬라이드 3~5장 준비', '9/10(토)까지 회신'], 'deadline': '2023-09-10', 'priority': 3}


In [29]:
# Step 4-② 한 곳만 바꾸기 — description 품질이 결과를 바꿉니다
# TODO(🔰): EmailSummaryV2의 priority description을 더 구체적으로 바꾸세요
# 지금: "중요도 1~5"  →  언제 1이고 언제 5인지 AI가 알 수 없습니다
# 힌트: 📖 강의 교안 모듈 3-3 "Field(description=...)이 왜 중요한가?"
# 바꾸기 전(result.priority)과 후를 비교해보세요

class EmailSummaryV2(BaseModel):
    sender:       str           = Field(description="발신자 이름")
    purpose:      str           = Field(description="이메일 목적 한 문장")
    action_items: list[str]     = Field(description="처리 필요 항목 목록")
    deadline:     Optional[str] = Field(default=None, description="기한. 없으면 None")
    priority:     int           = Field(
        description="기본이 1, purpose가 있으면 중요도 1 증가, 발신자 이름이 있으면 중요도 1 증가, '회신'이라는 단어가 들어가면 중요도 1 증가",   
        ge=1, le=5
    )
# TODO(🔰): 구체적으로 바꾸세요 - 해보니까 3이 아닌 다른 결과를 내는 description을 찾기 쉽지 않음 - 모델이 너무 단순하게 구성되어 있기 때문 (tool을 붙이면 점점 똑똑해짐)
# '회신'이라는 단어가 들어가면 중요도 5 - 얘는 잘 작동함 
# 발신자 이름이 있으면 중요도 4 - 얘는 잘 작동함
# 2023년 9월 9일과 비교했을 때 날짜가 이미 지났으면 많이 지날수록 중요도가 낮고, 지나지 않았으면 그 날짜가 다가올수록 중요도가 높음. - 날짜를 인식 못해서 날짜 있는 지시사항은 1
# 처리 필요 항목 목록 개수 - 잘 작동 안함 (3이 나와야 하는데 1로 나옴)
# 기본이 1, purpose가 있으면 중요도 1 증가, 발신자 이름이 있으면 중요도 1 증가, '회신'이라는 단어가 들어가면 중요도 1 증가 - 이렇게 여러 개 넣으면 정신 못 차림

chain_v2 = prompt_template | llm.with_structured_output(EmailSummaryV2)
result_v2 = chain_v2.invoke({"email": test_email})

print(f"수정 전 priority: {result.priority}")
print(f"수정 후 priority: {result_v2.priority}")
print("\n(같은 이메일인데 description이 다르면 우선순위가 달라지나요?)")

수정 전 priority: 3
수정 후 priority: 1

(같은 이메일인데 description이 다르면 우선순위가 달라지나요?)


---
## Step 4-③ PydanticOutputParser — with_structured_output 이전의 방법

📖 강의 연계: day3 강의교안 **모듈 3-3** "두 가지 구조화 방법 — 비교 먼저"

| | `PydanticOutputParser` | `with_structured_output` |
|---|---|---|
| **방식** | 프롬프트에 형식 지시문 삽입 → 파싱 | 모델 수준에서 형식 강제 |
| **안정성** | 가끔 형식 이탈 가능 | 더 안정적 (Function Calling 활용) |
| **호환성** | 모든 LLM | Function Calling 지원 모델만 (gpt-4o-mini ✅) |
| **프롬프트** | `{format}` 자리에 지시문 삽입 필요 | 프롬프트 변경 불필요 |

> ✅ `get_format_instructions()` 가 스키마를 보고 "이렇게 출력해라"는 지시문을 자동 생성합니다.  
> 두 방식의 **출력 타입은 동일** — 모두 Pydantic 인스턴스 반환.

In [30]:
# Step 4-③ PydanticOutputParser — 두 방식 비교 체험
from langchain_core.output_parsers import PydanticOutputParser

class EmailSummaryPOP(BaseModel):
    sender:  str = Field(description="발신자 이름")
    subject: str = Field(description="메일 제목")
    summary: str = Field(description="본문 3문장 이내 요약")
    date:    str = Field(description="미팅 날짜·시간. 없으면 빈 문자열")

# ─── 방법 ①: PydanticOutputParser ─────────────────────────────────────
parser_pop          = PydanticOutputParser(pydantic_object=EmailSummaryPOP)
format_instructions = parser_pop.get_format_instructions()

print("📋 get_format_instructions() 자동 생성 지시문 (앞 150자):")
print(format_instructions[:150] + "...")

# {format} 자리에 지시문을 미리 고정
prompt_pop = ChatPromptTemplate.from_messages([
    ("system", "이메일에서 핵심 정보를 추출해. 모르면 빈 문자열로 두고 추측하지 마."),
    ("human", "아래 형식만 지켜 JSON으로 출력해.\n{format}\n\n이메일:\n{email_raw}"),
]).partial(format=format_instructions)

chain_pop  = prompt_pop | llm | parser_pop
result_pop = chain_pop.invoke({"email_raw": test_email})

print(f"\n✅ PydanticOutputParser 결과:")
print(f"   type:    {type(result_pop).__name__}")
print(f"   sender:  {result_pop.sender}")
print(f"   date:    {result_pop.date}")

# ─── 방법 ②: with_structured_output (오늘 메인) ────────────────────────
result_wso = chain_struct.invoke({"email": test_email})   # Step 4-①에서 만든 체인 재사용

print(f"\n✅ with_structured_output 결과:")
print(f"   type:    {type(result_wso).__name__}")
print(f"   sender:  {result_wso.sender}")
print(f"   date:    {result_wso.deadline}")

print("\n💡 둘 다 Pydantic 인스턴스 반환 — 접근 방식은 동일합니다")
print("   with_structured_output이 더 간결하고 안정적 → 이 과정의 표준")

📋 get_format_instructions() 자동 생성 지시문 (앞 150자):
The output should be formatted as a JSON instance that conforms to the JSON schema below.

As an example, for the schema {"properties": {"foo": {"titl...

✅ PydanticOutputParser 결과:
   type:    EmailSummaryPOP
   sender:  홍길동
   date:    9/12(수) 오전 10시

✅ with_structured_output 결과:
   type:    EmailSummary
   sender:  홍길동
   date:    2023-09-10

💡 둘 다 Pydantic 인스턴스 반환 — 접근 방식은 동일합니다
   with_structured_output이 더 간결하고 안정적 → 이 과정의 표준


---
## 연습문제 1 — IT 용어 사전 (RunnablePassthrough + batch)

📖 강의 연계: day3 강의교안 **모듈 3-1** "⭐ Runnable 4종" → RunnablePassthrough

**설계 포인트**
- `{"term": RunnablePassthrough()}`: `chain.invoke("용어")` 처럼 문자열 직접 입력 가능
- `system` 메시지에 `'용어: 설명'` 형식을 강제 → 파싱이나 UI 표시가 쉬워짐
- `batch(terms)`: 여러 용어를 한 번에 처리 (내부적으로 병렬 처리)

In [31]:
# 연습문제 1-① 그대로 실행 — IT 용어 사전 체인
# 📖 강의 연계: day3 강의교안 모듈 3-1 "⭐ Runnable 4종" RunnablePassthrough

# system: AI 역할·제약·출력 형식을 한 번에 지정
# '용어: 설명' 형식을 강제하면 파싱이나 UI 표시가 편해집니다
prompt_dict = ChatPromptTemplate.from_messages([
    ("system", (
        "너는 IT 관련 용어를 설명해주는 도우미다. "
        "답변은 한국어로. 간결하고 정확하게. "
        "형식은 반드시 '용어: 설명' 형태로만 작성하라."
    )),
    ("human", "다음 IT 용어를 설명하라: {term}"),
])

# {"term": RunnablePassthrough()}: invoke("대역폭") → {"term": "대역폭"} 자동 변환
chain_dict = {"term": RunnablePassthrough()} | prompt_dict | llm | StrOutputParser()

# 단일 조회
print("✅ 단일 조회:")
print(chain_dict.invoke("대역폭"))   # 문자열 직접 입력!

# batch로 여러 용어 동시 처리
terms_it = ["API", "클라우드", "컨테이너", "마이크로서비스"]
results_it = chain_dict.batch(terms_it)
print("\n✅ batch 처리:")
for term, r in zip(terms_it, results_it):
    print(f"  [{term}]")
    print(f"  {r[:80]}...")

✅ 단일 조회:
대역폭: 네트워크에서 데이터 전송이 가능한 최대 속도를 나타내며, 일반적으로 초당 전송할 수 있는 데이터의 양(비트 또는 바이트)으로 측정된다.

✅ batch 처리:
  [API]
  API: 애플리케이션 프로그래밍 인터페이스(Application Programming Interface)의 약자로, 소프트웨어 간의 상호작용을 ...
  [클라우드]
  클라우드: 인터넷을 통해 데이터 저장, 처리 및 관리 서비스를 제공하는 기술로, 사용자는 물리적인 서버나 저장 장치 없이도 필요한 리소스를 원격...
  [컨테이너]
  컨테이너: 소프트웨어를 실행하기 위한 경량화된 실행 환경으로, 애플리케이션과 그 의존성을 패키징하여 격리된 상태로 배포하고 실행할 수 있게 해주...
  [마이크로서비스]
  마이크로서비스: 애플리케이션을 독립적인 작은 서비스로 나누어 개발하는 아키텍처 스타일로, 각 서비스가 특정 기능을 수행하며 서로 통신하여 전체 ...


In [43]:
# 연습문제 1-② 한 곳만 바꾸기 — 특정 분야 전문 용어 사전으로 전환
# TODO(🔰): system 메시지를 수정해 분야를 전문화하세요
# 예시: "AI 용어", "네트워크 보안 용어", "클라우드 컴퓨팅 용어"
# 힌트: "IT 관련 용어" 부분만 바꾸면 됩니다

prompt_specialized = ChatPromptTemplate.from_messages([
    ("system", (
        "너는 운동 전문 용어를 설명해주는 도우미다. "   # TODO(🔰): ___ 채우기
        "답변은 한국어로. 간결하고 정확하게. "
        "형식은 반드시 '용어: 설명' 형태로만 작성하라."
    )),
    ("human", "다음 용어를 설명하라: {term}"),
])
chain_specialized = {"term": RunnablePassthrough()} | prompt_specialized | llm | StrOutputParser()

# 같은 용어를 두 체인으로 비교해보세요
test_term = "사이드 레터럴 레이즈"
print(f"=== 일반 IT 용어 사전 [{test_term}] ===")
print(chain_dict.invoke(test_term))

print(f"\n=== 전문화된 운동 용어 사전 [{test_term}] ===")
# TODO(🔰): chain_specialized.invoke(test_term) 을 실행해 차이를 확인하세요
print(chain_specialized.invoke(test_term))

=== 일반 IT 용어 사전 [사이드 레터럴 레이즈] ===
사이드 레터럴 레이즈: 어깨 근육을 강화하기 위한 운동으로, 양팔을 옆으로 들어 올려 측면 삼각근을 자극하는 동작이다. 주로 덤벨을 사용하여 수행하며, 어깨의 넓이와 힘을 증가시키는 데 효과적이다.

=== 전문화된 운동 용어 사전 [사이드 레터럴 레이즈] ===
사이드 레터럴 레이즈: 어깨의 측면 근육인 삼각근을 강화하기 위해 팔을 옆으로 들어 올리는 운동. 주로 덤벨을 사용하여 수행하며, 어깨의 넓이를 증가시키는 데 효과적이다.


---
## 연습문제 2 — 고객 요청 자동 구조화 (Literal + list[str] + with_structured_output)

📖 강의 연계: day3 강의교안 **모듈 3-4** "심화 ③ Literal + list[str] 타입 실전"

**핵심 설계 포인트**

| 타입 | 의미 | 예시 |
|------|------|------|
| `str` | 자유 텍스트 | 핵심문제, 요청사항 |
| `Literal["A","B"]` | 정해진 값 중 하나만 허용 | 분류 카테고리 |
| `list[str]` + `min_length` | 최소 N개 강제 | 태그 3~5개 |
| `...`(Ellipsis) | 필수 필드 | None 허용 안 됨 |

> ⚠️ **Literal 설계 원칙**: `"기타"` 항목을 반드시 포함하세요.  
> 없으면 LLM이 가장 가까운 범주를 억지로 선택하거나 오류가 발생합니다.

In [44]:
# 연습문제 2-① 그대로 실행 — 고객 요청 자동 구조화
import json
from typing import Literal

class TicketSummary(BaseModel):
    핵심문제: str = Field(..., description="고객이 겪는 핵심 문제 (명사형 어미로)")
    요청사항: str = Field(..., description="고객이 원하는 조치/해결 (명사형 어미로)")

    # Literal: 이 6개 값 중 하나만 허용 — LLM이 다른 카테고리를 임의 생성 불가
    분류: Literal["문제해결", "기능요청", "결제/계정", "성능", "문의/가이드", "기타"] = Field(
        description="6개 유형 중 하나만 선택"
    )

    # list[str] + min/max_length: 태그 개수를 3~5개로 강제
    태그: list[str] = Field(
        description="소문자, 언더바(_) 사용. 예: ['로그인_실패', 'ios']",
        min_length=3, max_length=5,
    )
    에스컬레이션: bool = Field(description="상위 부서 에스컬레이션 필요 여부")

SYSTEM_TICKET = (
    "너는 고객 요청 요약 전문가다. 한국어로 답하고 아래 기준을 지켜라.\n"
    "- 분류: 6개 중 하나만 선택\n"
    "- 태그: 3~5개, 소문자, 언더바(_) 사용\n"
    "- 정보 부족 시 '미상' 사용, 개인정보 미포함"
)
prompt_ticket = ChatPromptTemplate.from_messages([
    ("system", SYSTEM_TICKET),
    ("human", "다음 고객 요청을 구조화해줘:\n{request}"),
])
chain_ticket = prompt_ticket | llm.with_structured_output(TicketSummary)

ticket_text = "어제부터 앱 로그인에 계속 실패합니다. 비밀번호를 새로 바꿨는데도 안되네요. 아이폰입니다."
result = chain_ticket.invoke({"request": ticket_text})

print("✅ 구조화 결과:")
print(json.dumps(result.model_dump(), ensure_ascii=False, indent=2))

# 여러 티켓을 batch로 동시 처리
tickets = [
    "결제가 두 번 됐어요. 환불해주세요.",
    "5G인데 자꾸 LTE로 바뀝니다.",
]
results = chain_ticket.batch([{"request": t} for t in tickets])
print("\n✅ batch 처리 결과:")
for r in results:
    print(f"  [{r.분류}] {r.핵심문제}")

✅ 구조화 결과:
{
  "핵심문제": "앱 로그인 실패",
  "요청사항": "비밀번호 변경 후에도 로그인 불가",
  "분류": "기타",
  "태그": [
    "앱_로그인",
    "비밀번호_변경",
    "아이폰",
    "로그인_문제"
  ],
  "에스컬레이션": false
}

✅ batch 처리 결과:
  [결제/계정] 결제가 두 번 됨
  [기능요청] 5G에서 LTE로 자주 변경됨


In [48]:
# 연습문제 2-② 한 곳만 바꾸기 — 내 서비스 도메인 분류기로 확장
# TODO(🔰): TicketSummaryV2의 Literal 카테고리를 내 도메인에 맞게 바꾸세요
# 예시: 쇼핑몰 → ["배송문의", "반품교환", "결제오류", "상품불량", "기타"]
#       HR 챗봇  → ["급여", "휴가", "복리후생", "계약", "기타"]
#       IT 헬프데스크 → ["계정", "네트워크", "소프트웨어", "하드웨어", "기타"]
# 힌트: Literal[...] 안의 카테고리명만 바꾸면 됩니다 — '기타' 항목은 필수!

class TicketSummaryV2(BaseModel):
    운동종류: str = Field(..., description="운동 종류(명사형 어미로)")
    운동부위: Literal[
        "가슴", "어깨", "등", "하체", "기타"   # TODO(🔰): 내 도메인 카테고리로 교체 (기타 필수!)
    ] = Field(description="운동 부위 (명사형 어미로)")
    태그:    list[str] = Field(description="소문자 태그 3~5개", min_length=3, max_length=5)
    난이도: Literal["하", "중", "상"] = Field(description="헬스를 시작한지 3개월 된 사람 기준으로 난이도 판단")

prompt_v2   = ChatPromptTemplate.from_messages([
    ("system", "너는 운동 전문 트레이너다."),
    ("human",  "{request}"),
])
chain_v2 = prompt_v2 | llm.with_structured_output(TicketSummaryV2)

# TODO(🔰): 아래 코드를 실행해 결과를 확인하고 분류가 제대로 되는지 검토하세요
result_v2 = chain_v2.invoke({"request": "몬스터글루트"})
print(result_v2.model_dump())

{'운동종류': '근력운동', '운동부위': '하체', '태그': ['스쿼트', '레그프레스', '데드리프트', '런지', '힙쓰러스트'], '난이도': '중'}


---
## 연습문제 3 — 예의바른 번역기 (partial_variables + RunnablePassthrough + stream)

📖 강의 연계: day3 강의교안 **모듈 3-4** "심화 ④ partial_variables + stream"

**설계 포인트**
- `prompt.partial(...)`: 기본 언어 방향을 미리 고정해두고, 필요 시 `invoke()` 에서 override
- `RunnablePassthrough`: `chain.invoke("문자열")` 직접 입력 패턴
- `stream`: 긴 번역 결과를 실시간 출력 — 체감 응답속도가 크게 향상됨

> 💡 **partial override 원칙**: `partial()`로 고정한 변수는 `invoke()` 시 같은 키로  
> 값을 넣으면 덮어씌워집니다. 기본값을 설정하면서도 유연하게 바꾸는 강력한 패턴입니다.

In [52]:
# 연습문제 3-① 그대로 실행 — partial + stream 번역기
from langchain_core.runnables import RunnablePassthrough

prompt_translator = ChatPromptTemplate.from_messages([
    ("system", (
        "너는 번역기야. {input_language}를 {output_language}로 번역해줘. "
        "자연스럽게, 비즈니스 어조로. 이모티콘 등 비즈니스에 맞지 않는 요소는 제거해."
    )),
    ("human", "다음 문장을 번역해줘: {text}"),
])

# partial로 기본 언어 방향 고정 — invoke 시 같은 키로 값을 넣으면 override 가능
prompt_translator = prompt_translator.partial(
    input_language="영어",
    output_language="한국어",
)

# RunnablePassthrough: chain.invoke("문자열") 직접 입력 패턴
chain_translator = {"text": RunnablePassthrough()} | prompt_translator | llm | StrOutputParser()

# ─── 기본 방향 (한국어→영어) ──────────────────────────────────────────
ko_text = "안녕하세요. 내일 미팅이 취소됐습니다. 일정을 다시 조율해 주세요."
print("✅ 기본 방향 (한국어→영어), invoke:")
print(chain_translator.invoke(ko_text))

# ─── stream + 언어 방향 override ──────────────────────────────────────
en_casual = "today was great! meeting moved to 3pm — any issues just let me know asap"
print("\n✅ stream + override (영어→한국어):")
for chunk in chain_translator.stream({
    "text":            en_casual,
    "input_language":  "영어",    # partial 기본값 override
    "output_language": "한국어",
}):
    print(chunk, end='', flush=True)
print()

✅ 기본 방향 (한국어→영어), invoke:
안녕하세요. 내일 미팅이 취소되었습니다. 일정을 다시 조율해 주시기 바랍니다.

✅ stream + override (영어→한국어):
오늘은 정말 좋았습니다! 회의가 오후 3시로 변경되었습니다. 문제가 있으면 가능한 한 빨리 알려주시기 바랍니다.


In [53]:
# 연습문제 3-② 한 곳만 바꾸기 — 번역 스타일 규칙 추가
# TODO(🔰): system 메시지에 번역 규칙을 한 가지 추가해보세요
# 예시 규칙:
#   "기술 용어는 영어 그대로 유지해."
#   "모든 문장을 존댓말로 번역해."
#   "문장이 길면 두 문장으로 나눠도 좋아."
#   "숫자는 항상 한자리 이상 남기고 반올림해."
# 비교: 같은 입력을 chain_translator와 chain_custom에서 실행해 차이를 확인하세요

prompt_custom = ChatPromptTemplate.from_messages([
    ("system", (
        "너는 번역기야. {input_language}를 {output_language}로 번역해줘. "
        "자연스럽게, 비즈니스 어조로. 이모티콘 등 비즈니스에 맞지 않는 요소는 제거해. "
        "마땅한 한국어 단어가 없는 경우 영어 발음으로 번역해줘."   # TODO(🔰): 규칙 하나 추가 (위 예시 참고)
    )),
    ("human", "다음 문장을 번역해줘: {text}"),
]).partial(input_language="영어", output_language="한국어")

chain_custom = {"text": RunnablePassthrough()} | prompt_custom | llm | StrOutputParser()

# 비교 실행
test_text = "The first sprint of the AI Campus project will be completed on August 14th."
print("=== 기본 번역기 ===")
print(chain_translator.invoke(test_text))

print("\n=== 규칙 추가된 번역기 ===")
print(chain_custom.invoke(test_text))# TODO(🔰): chain_custom.invoke(test_text) 를 실행해 차이를 확인하세요

=== 기본 번역기 ===
AI 캠퍼스 프로젝트의 첫 번째 스프린트는 8월 14일에 완료될 예정입니다.

=== 규칙 추가된 번역기 ===
AI 캠퍼스 프로젝트의 첫 번째 스프린트는 8월 14일에 완료될 예정입니다.


---
## 🔰 기본 미션 — 내 서비스 조각에 적용

📖 강의 연계: day3 강의교안 **모듈 3-4** "마이 서비스 조각 적용 실습"

> Day 2에서 선언한 내 서비스 아이디어의 출력 구조를 Pydantic으로 정의하고,  
> `with_structured_output` 체인을 만들어 **테스트 3케이스를 실행**합니다.
>
> 📁 파일 연속성: `Day 2의 my_service_v2.py` 프롬프트를 아래 셀에 그대로 활용하세요.

**필요한 것:**
1. 내 서비스 출력 스키마 (Pydantic BaseModel, 필드 3개 이상)
2. 시스템 프롬프트 + ChatPromptTemplate
3. 체인: `prompt | llm.with_structured_output(내_모델)`
4. 테스트 입력 3개 실행 → LangSmith 링크 제출

> 아래 3가지 서비스 예시 중 하나를 참고하거나, 내 아이디어를 직접 구현하세요.

In [58]:
# ── 서비스 예시 참고 (내 아이디어에 맞게 수정하세요) ───────────────────────
#
# 예시 A (회의록 요약기):
#   key_decisions: list[str]  = Field(description="주요 결정 사항 목록")
#   action_items:  list[str]  = Field(description="처리 필요 항목 (담당자+기한 포함)")
#   summary_3lines: str       = Field(description="전체를 3줄 이내로 요약")
#
# 예시 B (이메일 초안 도우미):
#   subject:    str = Field(description="이메일 제목")
#   body:       str = Field(description="본문 내용")
#   tone_score: int = Field(description="어조 공손함 1(캐주얼)~5(격식체)", ge=1, le=5)
#
# 예시 C (민원 분류기):
#   category:   Literal["배송문의","제품불량","환불교환","계정결제","기타"]
#   priority:   Literal["높음","보통","낮음"]
#   suggested_response: str = Field(description="고객에게 전달할 1줄 답변 초안")
# ────────────────────────────────────────────────────────────────────────────

# TODO(🔰): 내 서비스 출력 스키마를 정의하세요 (필드 3개 이상)
class News_to_Youtube(BaseModel):
    title: str = Field(description= "유튜브 이용자들이 관심을 가질 만한 제목")
    topic: list[str] = Field(description="기사의 핵심 쟁점을 3가지 이내로 정리")
    background: str = Field(description= "유튜브 이용자들이 기사의 내용을 이해하기 위해 필요한 배경지식 제시")
    script: str = Field(description= "실제 유튜버가 말하듯이 구어체로 작성된 대본") 
    view_predict: int = Field(description="구독자 3만명 유튜버 기준 이 영상의 예상 조회수")   


In [59]:
# TODO(🔰): Day 2에서 작성한 시스템 프롬프트를 아래에 붙여넣고 체인을 구성하세요
MY_SYSTEM_PROMPT = """
당신은 유튜브 대본 작성 전문가입니다. 기사 내용이 입력되면 그 내용을 기반으로 유튜브 대본을 작성하세요.
"""   

my_prompt = ChatPromptTemplate.from_messages([
    ("system", MY_SYSTEM_PROMPT),
    ("human", "{user_input}"),
])
my_chain = my_prompt | llm.with_structured_output(News_to_Youtube)

# TODO(🔰): 내 서비스에 맞는 실제 테스트 입력 3개로 바꾸세요
test_inputs = [
    """ 
    예비부모들이 출산을 앞두고 떠나는 '태교여행'의 트렌드가 변화하고 있다. 이전에는 괌이나 동남아시아 같은 휴양지를 택했다면, 이제는 도쿄 오사카 등 일본 도심으로 떠나는 여행이 부상하면서다. 일본은 이동시간이 짧고 음식도 비교적 한국인 입맛에 잘 맞아 태교여행지로 선호되는 국가이긴 했지만, 그간 휴양지인 오키나와만 각광받았던 것과도 사뭇 다른 양상이다.
    9일 한국일보 취재를 종합하면, 최근 들어 국내 포털 사이트나 사회관계망서비스(SNS)에서는 일본에서 육아용품 구매에 성공한 후기글이나 영상을 쉽게 찾아볼 수 있다. 일본에서 가장 유명한 육아용품 전문점인 '아카짱혼포'는 한국인 예비부모 사이에서 '필수 코스'로 자리 잡은 지 오래다. 심지어 구매한 상품을 안전하게 운반하기 위해선 일본 내 어떤 상점에서 완충재를 사서 포장하면 된다거나, 바퀴 달린 화분받침을 구매해 박스에 부착하면 공항까지 이동이 편하다는 등의 비법이 공유되기도 한다.
    예비부모들이 안락한 휴양지를 제쳐놓고 번화한 일본 도심을 찾는 이유는 육아용품을 쇼핑하기에 좋은 환경이라는 이유가 크다. 일본 특유의 다양한 캐릭터 제품이나 기발한 상품들도 매력적이지만, 신생아 유모차나 식탁 의자 등 한국에선 품절대란으로 구하기 어려운 인기 제품들이 일본에선 비교적 구하기 쉽다. 무엇보다도 엔저(엔화 약세) 현상에 면세 혜택까지 중복 할인 효과가 있어 현지 구매가 비용 면에서 상당한 이점이 있다.
    이런 현상은 태교여행에 대한 인식 변화와 맞물려 있다. 이전에는 '임신부의 휴식'이나 '예비부부가 누릴 마지막 자유'라는 향유적 목적이 태교여행의 중심을 차지했다면, 이제는 육아용품을 더 저렴하게 구매하는 것과 같은 실용적 목적이 커진 셈이다. 임신 20주차에 후쿠오카로 태교여행을 갔다온 김지영(27)씨는 "엔화도 저렴하고 아기용품으로 유명한 가게가 있어서 일본을 (태교여행지로) 선택하게 됐다"면서 "한국에 없는 아기용품도 있고 면세도 받을 수 있었다"고 만족감을 드러냈다.
    전문가들은 이를 정보화 세대의 특징으로 설명했다. 정보화 시대에서는 주어진 정보를 어떻게 활용하느냐에 따라 보다 합리적 소비가 가능하기 때문에, 소비 과정 자체에서 만족을 찾는 경우가 많다는 것이다. 이은희 인하대 소비자학과 교수는 "최근 젊은 세대들은 적극적으로 정보를 찾아내서 좋은 상품을 합리적으로 손에 넣는 과정 자체를 중요하게 생각한다"면서 "특히 육아용품은 최근 프리미엄화돼서 장만하기가 쉽지 않은 만큼, 실용성과 합리성을 추구하는 소비 과정의 의미가 커질 수밖에 없다"고 분석했다.
    """,   
    """아침에 지하철을 타러 가던 중 평소와 다른 안내 문구가 눈에 들어왔다. 역사 곳곳에 붙어 있는 ‘기후동행카드 30일권 운영 종료’ 안내와 함께 ‘모두의카드(K-패스)에서 혜택을 받으라’는 내용이었다. 평소 기후동행카드를 이용하고 있었던 만큼 갑작스러운 안내가 조금 의아했다. ‘기후동행카드가 없어지는 것인가?’, ‘모두의카드는 또 무엇일까?’라는 궁금증이 생겨 관련 내용을 찾아봤다. 
    확인해 보니 기후동행카드 전체가 사라지는 것이 아니라, 30일권 운영이 종료되고 새로운 ‘기후동행패스’로 대체되는 셈이다. 바로 9월 1일부터 정부의 ‘모두의카드(K-패스)'에 서울시의 특화 혜택을 더한 ‘모두의카드(기후동행패스)’가 본격적으로 시행되었다. 기존 기후동행카드 이용자를 위한 '기후동행패스' 실물 카드 무상 교환 이벤트도 함께 진행되고 있었다. 직접 실물 카드 교환 현장을 찾아가 확인해 봤다. 
    기존 기후동행카드를 새로운 카드로 직접 교환하기 위해 서울역 홍보부스를 찾았다. 이번 현장 이벤트는 서울역을 비롯해 성수, 건대입구, 시청, 연신내, 노원, 홍대입구, 잠실, 사당, 선릉, 여의도, 가산디지털단지 등 서울 주요 지하철역 12곳에서 진행된다.
    서울역 홍보부스는 지하 1층 대합실 구 안내부스 옆 공간에 마련돼 있었다. 지하철을 이용해 서울역에 도착한 뒤 안내를 따라가니 어렵지 않게 찾을 수 있었다.
    현장에 도착하자 안내원이 기존 기후동행카드 삭제부터 새로운 카드 교환까지 절차를 설명했다. 먼저 기존에 사용하던 기후동행카드를 삭제해야 했는데, 이 과정에서 티머니 카드&페이 누리집에 접속해 로그인이 필요했다.
    여기서 현장 방문 전에 미리 준비하면 좋은 부분이 있었다. 티머니 카드&페이 누리집 로그인 과정에서 아이디나 비밀번호를 확인하느라 시간이 오래 걸릴 수 있기 때문이다. 따라서 카드 교환을 위해 홍보부스를 방문한다면 미리 티머니 카드&페이 누리집에 로그인해 두거나 아이디와 비밀번호를 확인해 두는 것이 좋다. 안내원의 설명에 따라 기존 카드를 삭제한 뒤 새로운 기후동행패스 실물 카드로 교환을 마쳤다.
    ‘모두의카드(기후동행패스)’는 정부의 모두의카드(K-패스)에 서울시 특화 혜택을 더한 교통 서비스다. 기존 기후동행카드와 달리 전국 대중교통 이용과 환급 혜택을 받을 수 있으며, GTX와 신분당선, 광역버스 등도 환급 대상에 포함된다.
    이번 변화에서 가장 눈에 띄는 부분은 청년 할인 대상 연령 확대다. 기존 만 19~34세였던 청년 할인 대상이 만 19~39세까지 넓어졌다. 30대 중후반도 확대된 기준에 따라 환급 혜택을 받을 수 있게 됐다.
    기존 모두의카드(K-패스)를 이용하고 있는 시민이라면 별도로 카드를 변경할 필요도 없다. K-패스 회원가입 과정에서 서울시 거주지 인증을 완료했다면 서울시 특화 혜택이 자동으로 적용된다. 월별 대중교통 이용 금액을 합산해 정률형과 정액형 가운데 이용자에게 더 유리한 방식으로 환급 혜택이 제공된다.
    기후동행패스 현장 홍보부스에서는 기존 기후동행카드 실물 카드(3,000원)를 기후동행패스 실물 카드(4,000원)로 무상 교환할 수 있다. 기존 이용자가 별도로 새로운 실물 카드를 구매하는 것이 아니라, 기존 카드를 가지고 현장 홍보부스를 방문해 교환하는 방식이다. 평소 기후동행카드를 이용하던 시민이라면 복잡한 절차를 거칠 필요 없이 현장에서 안내를 받으며 쉽게 교환을 진행할 수 있다.
    현장 홍보부스에서는 카드 교환 외에도 디지털 기기 사용이 익숙하지 않은 시민을 위한 지원도 제공된다. 어르신 등을 대상으로 모바일 앱 설치부터 K-패스 회원가입, 카드 등록까지 일대일로 안내해 온라인으로 직접 진행하기 어려운 시민들이 현장에서 도움을 받을 수 있도록 한 것이다.
    기후동행패스 현장 홍보부스는 10월 2일까지 일요일과 공휴일을 제외하고 운영된다. 평일에는 오전 7시부터 11시, 오후 4시부터 8시까지 운영하며, 주말에는 오전 10시부터 오후 6시까지 운영한다. 서울역 외에도 성수역, 건대입구역, 시청역, 연신내역, 노원역, 홍대입구역, 잠실역, 사당역, 선릉역, 여의도역, 가산디지털단지역 등 총 12개 역사에서 홍보부스를 운영한다.""",
    """개인정보 유출 사태 이후 소비자 이탈 우려가 제기됐던 쿠팡의 월간 카드 결제액이 5조원대를 돌파한 것으로 나타났다. 이용자 수도 최고 수준으로 늘어났다. 신속한 배송 체계를 바탕으로 한 생필품·식품의 반복 구매 구조와 멤버십의 락인(Lock-In) 효과가 결제액 증가로 이어진 것으로 풀이된다.
    9일 AI 데이터 테크 기업 아이지에이웍스의 모바일인덱스에 따르면 쿠팡의 지난 7월 신용·체크카드 추정 결제액은 5조942억원으로 집계됐다.
    이는 전달의 4조8천337억원보다 5.4% 늘어난 것으로, 정보 유출 사태 이후 제공된 월간 집계 기준 최고치다.
    지난 8월 추정 결제액도 4조9301억원으로 5조원에 육박했다. 지난해 8월의 4조3777억원과 비교하면 12.6% 증가한 수치다.
    쿠팡 결제액은 올해 2월 4조219억원까지 내려갔으나 이후 꾸준한 회복세를 보였다. 로켓배송·새벽배송을 바탕으로 한 생필품·식품의 반복 구매 구조와 와우 멤버십 혜택이 결제액 증가에 영향을 미친 것으로 해석된다.
    또한 국내 이커머스 경쟁사들이 쿠팡의 배송망과 상품 구색, 구매 편의성을 대체할 만한 서비스를 제시하지 못한 점도 쿠팡 쏠림을 키운 요인으로 거론된다.
    쿠팡 이용자 수도 매달 꾸준히 늘었다. 
    지난 8월 쿠팡 앱의 월간 활성 이용자 수(MAU)는 3595만명으로 집계됐다. 지난 7월의 3544만명보다 51만명 가량 증가한 수준이다. 정보 유출 이전인 작년 10월 쿠팡의 MAU는 3438만명이었다.
    쿠팡의 지난달 MAU는 국내 별도 쇼핑 앱인 네이버플러스 스토어(904만명)의 약 4배 수준이다.
    지난해 11월 정보 유출 사태 직후 일부 소비자를 중심으로 회원 탈퇴 움직임이 있었지만, 실제 소비 단계에서는 이용자 이탈이 단기적으로 사실상 끝난 셈이다. 빠른 배송과 새벽배송, 멤버십 혜택이 일상 소비와 결합하면서 쿠팡 의존도가 높아졌다는 분석이 나온다.
    반면 토종 이커머스는 부진한 모습이다.
    G마켓의 지난 8월 추정 결제액은 2477억원으로 지난해 8월보다 25.9% 감소했고, 11번가도 2414억원으로 3.1% 줄었다.
    쿠팡의 지난 8월 결제액은 G마켓과 11번가 결제액을 합친 금액의 약 10.1배에 달했다.
    이번 결제액은 신용·체크카드 데이터를 토대로 AI 알고리즘이 추산한 수치로 실제 매출과 차이가 있을 수 있다. 네이버플러스 스토어 결제 데이터는 비교 대상에 포함되지 않았다.
    """
]

for i, text in enumerate(test_inputs, 1):
    try:
        r = my_chain.invoke({"user_input": text})
        print(f"\n=== 테스트 {i} ===")
        print(f"입력: {text}")
        print(f"결과: {r.model_dump()}")
    except Exception as e:
        print(f"\n=== 테스트 {i} 실패 ===")
        print(f"오류: {e}")
        print("💡 스키마를 단순화하거나 description을 더 명확하게 바꿔보세요")


=== 테스트 1 ===
입력:  
    예비부모들이 출산을 앞두고 떠나는 '태교여행'의 트렌드가 변화하고 있다. 이전에는 괌이나 동남아시아 같은 휴양지를 택했다면, 이제는 도쿄 오사카 등 일본 도심으로 떠나는 여행이 부상하면서다. 일본은 이동시간이 짧고 음식도 비교적 한국인 입맛에 잘 맞아 태교여행지로 선호되는 국가이긴 했지만, 그간 휴양지인 오키나와만 각광받았던 것과도 사뭇 다른 양상이다.
    9일 한국일보 취재를 종합하면, 최근 들어 국내 포털 사이트나 사회관계망서비스(SNS)에서는 일본에서 육아용품 구매에 성공한 후기글이나 영상을 쉽게 찾아볼 수 있다. 일본에서 가장 유명한 육아용품 전문점인 '아카짱혼포'는 한국인 예비부모 사이에서 '필수 코스'로 자리 잡은 지 오래다. 심지어 구매한 상품을 안전하게 운반하기 위해선 일본 내 어떤 상점에서 완충재를 사서 포장하면 된다거나, 바퀴 달린 화분받침을 구매해 박스에 부착하면 공항까지 이동이 편하다는 등의 비법이 공유되기도 한다.
    예비부모들이 안락한 휴양지를 제쳐놓고 번화한 일본 도심을 찾는 이유는 육아용품을 쇼핑하기에 좋은 환경이라는 이유가 크다. 일본 특유의 다양한 캐릭터 제품이나 기발한 상품들도 매력적이지만, 신생아 유모차나 식탁 의자 등 한국에선 품절대란으로 구하기 어려운 인기 제품들이 일본에선 비교적 구하기 쉽다. 무엇보다도 엔저(엔화 약세) 현상에 면세 혜택까지 중복 할인 효과가 있어 현지 구매가 비용 면에서 상당한 이점이 있다.
    이런 현상은 태교여행에 대한 인식 변화와 맞물려 있다. 이전에는 '임신부의 휴식'이나 '예비부부가 누릴 마지막 자유'라는 향유적 목적이 태교여행의 중심을 차지했다면, 이제는 육아용품을 더 저렴하게 구매하는 것과 같은 실용적 목적이 커진 셈이다. 임신 20주차에 후쿠오카로 태교여행을 갔다온 김지영(27)씨는 "엔화도 저렴하고 아기용품으로 유명한 가게가 있어서 일본을 (태교여행지로) 선택하게 됐다"면서 "한국에 없는 아기용품도 있고 면세도 받을 수 있었다"고

---
## ⭐ 심화 미션 — 중첩 모델 (MeetingMinutes)

📖 강의 연계: day3 강의교안 **모듈 3-3** "⭐ 심화: 중첩 모델"

아래 요구사항을 구현하세요. 힌트만 제공됩니다.

**요구사항:**
1. `ActionItem` 모델 정의 (`assignee: str`, `task: str`, `deadline: str`)
2. `MeetingMinutes` 모델 정의 — `action_items: list[ActionItem]` 포함
3. 아래 `meeting_text`로 체인 실행
4. `result.action_items[0].assignee` 처럼 **중첩 필드에 직접 접근**하세요

기본 미션 완료 후 팀 프로젝트 블록에 조기 합류할 수 있습니다.

In [39]:
# ⭐ 심화 구현 공간 — 아래 meeting_text를 분석하는 체인을 만드세요

meeting_text = """
일시: 8월 5일 수요일 오후 2시
참석: 홍길동(팀장), 김철수(개발), 이영희(기획)

결정 사항:
1. 8월 14일 발표 준비는 팀 전원이 참여한다.
2. SSE 스트리밍 기능은 필수 요건으로 확정한다.

액션 아이템:
- 홍길동: 8/10까지 발표 자료 초안 작성
- 김철수: 8/12까지 SSE 엔드포인트 구현
- 이영희: 8/11까지 서비스 플로우 다이어그램 완성
"""

# 여기에 구현하세요 ↓
# 1. ActionItem, MeetingMinutes 모델 정의
# 2. chain = prompt_template | llm.with_structured_output(MeetingMinutes)
# 3. result = chain.invoke({"email": meeting_text})  ← 프롬프트 변수 맞게 조정
# 4. print(result.action_items[0].assignee)  ← 중첩 필드 접근 확인

---
## ⭐ 심화 미션 ② — 내 경험 기반 자유 주제 구현

📖 강의 연계: day3 강의교안 **모듈 3-4** "마이 서비스 조각"

중첩 모델을 이해했다면, 이번엔 완전히 **내 것**을 만들어보세요.  
지금까지 살면서 겪은 불편함이나 반복 작업에서 아이디어를 찾아보세요.

| 내 상황 예시 | 구조화가 유용한 이유 |
|------------|------------------|
| 주간 업무 보고서 정리 | 핵심 지표·위험 신호·다음 액션을 자동 추출 |
| 독서·강의 노트 구조화 | 핵심 포인트·실천 교훈·복습 필요 여부 정리 |
| 고객·사용자 리뷰 분석 | 감정·핵심 불만·개선 제안으로 분류 |
| 면접 답변 피드백 | 강점·개선점·모범 답변 초안 생성 |
| 뉴스·공지 분류 | 카테고리·긴급도·관련 부서 자동 태깅 |

**구현 순서**: 상황 선택 → 스키마 설계 (필드 3개 이상) → 시스템 프롬프트 → 실제 텍스트로 테스트

In [40]:
# ⭐ 심화 미션 ②: 내 경험 기반 자유 주제 — 4단계 템플릿
# ════════════════════════════════════════════════════════════════

# ─── 1단계: 사용 사례 선언 ─────────────────────────────────────
# 한 줄로 "어떤 텍스트를 넣으면 무엇이 나오는 서비스인가"를 적으세요.
MY_USE_CASE = "TODO(⭐): 내 사용 사례를 한 줄로 적으세요"
# 예: "주간 보고서에서 핵심 지표와 위험 신호를 추출"
#     "수업 필기를 핵심 포인트 + 복습 필요 여부로 정리"
#     "고객 리뷰를 감정·불만·개선 제안으로 분류"

# ─── 2단계: 출력 스키마 설계 ─────────────────────────────────────
# 내 상황에서 꺼내야 할 '정보 조각'을 필드로 정의하세요.
# 설계 팁:
#   - 무엇을 추출해야 하나? (str, list[str], bool, int, Literal)
#   - Field(description=...)에 "언제 어떤 값인지"를 구체적으로 적어주세요
#   - 선택 필드(Optional)와 필수 필드를 구분하세요
class MyExperienceOutput(BaseModel):
    # TODO(⭐): 내 상황에 맞는 필드를 3개 이상 정의하세요
    pass   # ← 이 줄을 지우고 필드를 채우세요

# ─── 3단계: AI 전문가 역할 정의 ──────────────────────────────────
MY_EXPERT_ROLE = """
TODO(⭐): AI에게 줄 전문가 역할을 여기에 적으세요
예: "당신은 10년 경력의 HR 전문가입니다. 보고서를 읽고 핵심 지표를 추출합니다."
"""

my_exp_prompt = ChatPromptTemplate.from_messages([
    ("system", MY_EXPERT_ROLE),
    ("human", "{input_text}"),
])
my_exp_chain = my_exp_prompt | llm.with_structured_output(MyExperienceOutput)

# ─── 4단계: 실제 텍스트로 테스트 ─────────────────────────────────
# 내 실제 경험에서 나온 텍스트를 직접 붙여넣으세요.
# (회의록, 업무 메일, 책 메모, 리뷰, 뉴스 등 — 뭐든 가능합니다)
MY_TEST_INPUT = """
TODO(⭐): 처리하고 싶은 실제 텍스트를 여기에 붙여넣으세요
"""

try:
    result_exp = my_exp_chain.invoke({"input_text": MY_TEST_INPUT})
    print(f"✅ 내 경험 기반 구조화 성공!")
    print(f"   사용 사례: {MY_USE_CASE}")
    print(f"\n📦 구조화 결과:")
    for field, value in result_exp.model_dump().items():
        print(f"   {field}: {value}")
except Exception as e:
    print(f"❌ 오류: {e}")
    print("💡 팁: 스키마 필드를 3개로 줄이거나 description을 더 명확하게 바꿔보세요")

✅ 내 경험 기반 구조화 성공!
   사용 사례: TODO(⭐): 내 사용 사례를 한 줄로 적으세요

📦 구조화 결과:


---
## 📬 제출 & 자가 체크

### ✅ 완료 확인 목록

- [ ] **Step 1-④** `stream` 실행 시 토큰 단위 출력 확인, `max_concurrency` 옵션 적용
- [ ] **⭐ Step 1-⭐** Runnable 4종 (`Passthrough` / `Lambda` / `itemgetter` / `Sequence`) 전부 실행
- [ ] **Step 4-③** `PydanticOutputParser` 와 `with_structured_output` 두 방식 모두 실행·비교
- [ ] **연습문제 1** IT 용어 사전 — `RunnablePassthrough` + `batch` 동작 확인
- [ ] **연습문제 2** 고객 요청 요약봇 — `Literal` + `list[str]` + `with_structured_output` 동작 확인
- [ ] **연습문제 3** 번역기 — `partial` + `stream` + `override` 동작 확인
- [ ] **Step 1** LCEL 파이프 조립 + `invoke` / `batch` 실행
- [ ] **Step 2** TypedDict 런타임 동작 vs Pydantic ValidationError 비교 확인
- [ ] **Step 3** `TaskClassification` 인스턴스 생성 + `ValidationError` 의도적 발생
- [ ] **Step 4** LLM 응답 타입이 `EmailSummary`(str 아님)인 것을 확인
- [ ] **🔰 기본 미션** 내 서비스 스키마 3개 이상 필드 + 테스트 3케이스 실행
- [ ] **⭐ 심화 미션 ①** 중첩 모델(`MeetingMinutes`) 구현 및 중첩 필드 접근 확인
- [ ] **⭐ 심화 미션 ②** 내 경험 기반 주제 선택 → 스키마 설계 → 실제 텍스트로 테스트
- [ ] **TypedDict vs Pydantic** 차이를 한 문장으로 설명할 수 있다

### 📤 제출 방법

1. LangSmith([smith.langchain.com](https://smith.langchain.com)) 접속
2. 오늘 프로젝트의 기본 미션 트레이스 링크 복사
3. 슬랙 `#day3-제출` 채널에 링크 붙여넣기

> ℹ️ **조기 완료자**: ⭐ 심화 미션 ①② 구현 후 팀 프로젝트 블록에 바로 합류하세요.